<a href="https://colab.research.google.com/github/warrensuca/chinese-recipe-api/blob/main/the_woks_of_life_scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#The Woks of Life Recipe Scraper
This notebook scrapes recipe data from the Woks of Life website: https://thewoksoflife.com/category/recipes/:

For each recipe, we collect:

- Name
- Ingredients Full List
- Ingredients Names List
  - Used in clustering, very difficult to parse from full list
- Procedure
- Nutrition Facts

The final dataset is stored as a Pandas DataFrame and exported to CSV.

In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display
import time
print("setup complete")

setup complete


In [7]:
session = requests.Session()

session.headers.update({
    "User-Agent":
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
})

print("Session initialized")

Session initialized


#Scrape individual recipe URL
Exploration. Later, we will turn this into a function

In [30]:

url = "https://thewoksoflife.com/ayam-kecap-indonesian-braised-chicken/"
headings = [
      "Name",
      "Ingredients_List",
      "Ingredients_Names",
      "Procedure",
      "Nutrition_Facts"
  ]

response = session.get(url, timeout=10)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

recipe_data = {
    "Name": url.split("/")[-2] #getting title is an additional call, just clean later
}
print(recipe_data["Name"])
print('\n')

ingredients_list_raw = soup.find_all("li", class_ = "wprm-recipe-ingredient")
ingredients_list = []

for ingredient in ingredients_list_raw:
  ingredients_list.append(ingredient.get_text()[2:]) #remove a bullet point svg

print(ingredients_list)
print('\n')
recipe_data["Ingredients_List"] = ingredients_list

ingredients_names_raw = soup.find_all("span", class_ = "wprm-recipe-ingredient-name")
ingredients_names = []

for ingredient_name in ingredients_names_raw:
  ingredients_names.append(ingredient_name.get_text())

print(ingredients_names)
print('\n')
recipe_data["Ingredients_Names"] = ingredients_names

procedure_list_raw = soup.find_all("li", class_ = "wprm-recipe-instruction")
procedure_list = []

for procedure in procedure_list_raw:
  procedure_list.append(procedure.get_text())

print(procedure_list)
print('\n')
recipe_data["Procedure"] = procedure_list

nutrition_facts_list_raw = soup.find_all("span", class_ = "wprm-nutrition-label-text-nutrition-container")
nutrition_facts = []

servings = soup.find("span", class_ = "wprm-recipe-servings-adjustable-tooltip").get_text()
if(servings):
  nutrition_facts.append(servings)

for nutrition_fact in nutrition_facts_list_raw:
  #print(nutrition_fact.get_text())
  nutrition_facts.append(nutrition_fact.get_text())

print(nutrition_facts)

recipe_data["Nutrition_Facts"] = nutrition_facts

ayam-kecap-indonesian-braised-chicken


['2½ pounds chicken drumsticks', '½ teaspoon salt', '2 tablespoons neutral oil (such as canola, vegetable, or avocado oil)', '½ teaspoon ginger (finely minced or grated)', '3 cloves garlic (chopped)', '1/3 cup shallot or red onion (finely diced)', '3 dried chili peppers (optional)', '2¼ cups low sodium chicken stock', '¾ teaspoon galangal (or sand ginger powder)', '¾ teaspoon coriander powder', '¾ teaspoon ground or whole cloves', '3 dried Makrut lime leaves', '3 tablespoons kecap manis (Indonesian sweet soy sauce)', '2 tablespoons soy sauce', '1 teaspoon tamarind paste (optional)', '½ teaspoon ground white or black pepper']


['chicken drumsticks', 'salt', 'neutral oil', 'ginger', 'garlic', 'shallot or red onion', 'dried chili peppers', 'low sodium chicken stock', 'galangal', 'coriander powder', 'ground or whole cloves', 'dried Makrut lime leaves', 'kecap manis', 'soy sauce', 'tamarind paste', 'ground white or black pepper']


['In a large bowl,

#Create function based off exploration
This function will be used on each recipe of the site

In [36]:
def scrape_recipe(url: str):


  response = session.get(url, timeout=10)
  response.raise_for_status()

  soup = BeautifulSoup(response.text, "html.parser")

  recipe_data = {
      "Name": url.split("/")[-2] #getting title is an additional call, just clean later
  }


  ingredients_list_raw = soup.find_all("li", class_ = "wprm-recipe-ingredient")
  ingredients_list = []

  for ingredient in ingredients_list_raw:
    ingredients_list.append(ingredient.get_text()[2:]) #remove a bullet point svg


  recipe_data["Ingredients_List"] = ingredients_list


  ingredients_names_raw = soup.find_all("span", class_ = "wprm-recipe-ingredient-name")
  ingredients_names = []

  for ingredient_name in ingredients_names_raw:
    ingredients_names.append(ingredient_name.get_text())

  recipe_data["Ingredients_Names"] = ingredients_names


  procedure_list_raw = soup.find_all("li", class_ = "wprm-recipe-instruction")
  procedure_list = []

  for procedure in procedure_list_raw:
    procedure_list.append(procedure.get_text())



  recipe_data["Procedure"] = procedure_list

  nutrition_facts_list_raw = soup.find_all("span", class_ = "wprm-nutrition-label-text-nutrition-container")
  nutrition_facts = []

  servings_raw = soup.find("span", class_ = "wprm-recipe-servings-adjustable-tooltip")
  if servings_raw is not None:
    nutrition_facts.append(servings_raw.get_text())

  for nutrition_fact in nutrition_facts_list_raw:
    #print(nutrition_fact.get_text())
    nutrition_facts.append(nutrition_fact.get_text())

  recipe_data["Nutrition_Facts"] = nutrition_facts

  return recipe_data

In [24]:
recipe_data = scrape_recipe("https://thewoksoflife.com/pork-belly-char-siu/")

recipe_data

{'Name': 'pork-belly-char-siu',
 'Ingredients_List': ['2 pounds boneless skinless pork belly (if trimming your pork belly at home, save the bones and skins for soup!)',
  '3 tablespoons granulated sugar',
  '1½ teaspoons salt',
  '¼ teaspoon white pepper',
  '¼ teaspoon five spice powder',
  '1 tablespoon hoisin sauce',
  '1 tablespoon Shaoxing wine',
  '2 teaspoons light soy sauce',
  '½ teaspoon sesame oil',
  '1 clove garlic (minced)',
  '4 to 5 drops of red food coloring (optional)',
  '1 tablespoon maltose or honey',
  '1 tablespoon hot water'],
 'Ingredients_Names': ['boneless skinless pork belly',
  'granulated sugar',
  'salt',
  'white pepper',
  'five spice powder',
  'hoisin sauce',
  'Shaoxing wine',
  'light soy sauce',
  'sesame oil',
  'garlic',
  'drops of red food coloring',
  'maltose or honey',
  'hot water'],
 'Procedure': ['If you have a larger piece of pork belly, cut it lengthwise into long 1½- to 2-inch (4-5cm) thick strips.',
  'Make the marinade in a large bow

#Scrape single page
Exploration

In [25]:

response = session.get("https://thewoksoflife.com/category/recipes/page/{}/".format(1), timeout=10)
response.raise_for_status()

data = []

soup = BeautifulSoup(response.text, "html.parser")

recipes = soup.find_all(class_ = "wp-block-post-featured-image")


for recipe in recipes[:5]:

  data.append(scrape_recipe(recipe.a.get("href")))
data

[{'Name': 'shallot-vinaigrette',
  'Ingredients_List': ['1  large shallot (minced)',
   '2 tablespoons hot water',
   '¼ cup white wine vinegar',
   '2 tablespoons finely chopped chives',
   '1 garlic clove (minced)',
   '½ cup extra virgin olive oil',
   '2 teaspoons dijon mustard',
   '2 teaspoons honey',
   '1 teaspoon sea salt  or kosher salt',
   '¼ teaspoon freshly ground black pepper'],
  'Ingredients_Names': ['large shallot',
   'hot water',
   'white wine vinegar',
   'finely chopped chives',
   'garlic clove',
   'extra virgin olive oil',
   'dijon mustard',
   'honey',
   'sea salt',
   'freshly ground black pepper'],
  'Procedure': ['In a measuring cup, add the shallots and hot water. Then stir in the white wine vinegar, and let it sit while you prep the chives and garlic.',
   'To the measuring cup, add the olive oil, dijon mustard, honey, salt, pepper, and garlic. Whisk everything together until it’s emulsified. Add the chives, and whisk again. Give it a taste, and adjust

In [26]:
def scrape_page(page_num: int):

  response = session.get("https://thewoksoflife.com/category/recipes/page/{}/".format(page_num), timeout=10)
  response.raise_for_status()

  data = []

  soup = BeautifulSoup(response.text, "html.parser")

  recipes = soup.find_all(class_ = "wp-block-post-featured-image")


  for recipe in recipes:

    data.append(scrape_recipe(recipe.a.get("href")))

  return data

In [37]:
columns = [
      "Name",
      "Ingredients_List",
      "Ingredients_Names",
      "Procedure",
      "Nutrition_Facts"
  ]

temp_out = []

for page_num in range(1, 3):

  data = scrape_page(page_num)
  temp_out.extend(data)
#print(out)
df = pd.DataFrame(temp_out, columns=columns)

print(f"Total recipes scraped: {len(df)}")

display(df.head())

Total recipes scraped: 36


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts
0,shallot-vinaigrette,"[1 large shallot (minced), 2 tablespoons hot ...","[large shallot, hot water, white wine vinegar,...","[In a measuring cup, add the shallots and hot ...","[6, Calories: 173kcal (9%), Carbohydrates: 3g ..."
1,sesame-tofu,[2 blocks firm or extra firm tofu (anywhere fr...,"[firm or extra firm tofu, cornstarch, un-toast...","[Drain each block of tofu. Using your hands, g...","[6, Calories: 358kcal (18%), Carbohydrates: 29..."
2,pork-belly-char-siu,[2 pounds boneless skinless pork belly (if tri...,"[boneless skinless pork belly, granulated suga...","[If you have a larger piece of pork belly, cut...","[6, Calories: 682kcal (34%), Carbohydrates: 11..."
3,red-bean-zongzi,[2 pounds sweet rice (also known as sticky ric...,"[sweet rice, lye water, neutral oil, dried red...","[In a large, fine mesh strainer, rinse the swe...","[12, Calories: 344kcal (17%), Carbohydrates: 7..."
4,stir-fry-spinach,[1 pound spinach (preferably Taiwanese spinach...,"[spinach, neutral oil, garlic, Salt to taste, ...",[Wash spinach clean by soaking a few times and...,"[4, Calories: 130kcal (7%), Carbohydrates: 5g ..."


#Scrape the entire site!

In [38]:
pages = 80

out = []

for page_num in range(1, pages + 1):



    try:
      data = scrape_page(page_num)
      out.extend(data)

    except Exception as e:

      print(f"Error on page {page_num}: {e}")
      break
    print(f"page number: {page_num}")
    time.sleep(4) #be nice to the site, too aggresive can cause an exception


page number: 1
page number: 2
page number: 3
page number: 4
page number: 5
page number: 6
page number: 7
page number: 8
page number: 9
page number: 10
page number: 11
page number: 12
page number: 13
page number: 14
page number: 15
page number: 16
page number: 17
page number: 18
page number: 19
page number: 20
page number: 21
page number: 22
page number: 23
page number: 24
page number: 25
page number: 26
page number: 27
page number: 28
page number: 29
page number: 30
page number: 31
page number: 32
page number: 33
page number: 34
page number: 35
page number: 36
page number: 37
page number: 38
page number: 39
page number: 40
page number: 41
page number: 42
page number: 43
page number: 44
page number: 45
page number: 46
page number: 47
page number: 48
page number: 49
page number: 50
page number: 51
page number: 52
page number: 53
page number: 54
page number: 55
page number: 56
page number: 57
page number: 58
page number: 59
page number: 60
page number: 61
page number: 62
page number: 63
p

#Store Data in Pandas df
We can then get some insights and then export to CSV file

In [39]:
columns = [
      "Name",
      "Ingredients_List",
      "Ingredients_Names",
      "Procedure",
      "Nutrition_Facts"
  ]

df = pd.DataFrame(out, columns=columns)

print(f"Total recipes scraped: {len(df)}")

display(df.head())

Total recipes scraped: 1414


,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts
0,shallot-vinaigrette,"[1 large shallot (minced), 2 tablespoons hot ...","[large shallot, hot water, white wine vinegar,...","[In a measuring cup, add the shallots and hot ...","[6, Calories: 173kcal (9%), Carbohydrates: 3g ..."
1,sesame-tofu,[2 blocks firm or extra firm tofu (anywhere fr...,"[firm or extra firm tofu, cornstarch, un-toast...","[Drain each block of tofu. Using your hands, g...","[6, Calories: 358kcal (18%), Carbohydrates: 29..."
2,pork-belly-char-siu,[2 pounds boneless skinless pork belly (if tri...,"[boneless skinless pork belly, granulated suga...","[If you have a larger piece of pork belly, cut...","[6, Calories: 682kcal (34%), Carbohydrates: 11..."
3,red-bean-zongzi,[2 pounds sweet rice (also known as sticky ric...,"[sweet rice, lye water, neutral oil, dried red...","[In a large, fine mesh strainer, rinse the swe...","[12, Calories: 344kcal (17%), Carbohydrates: 7..."
4,stir-fry-spinach,[1 pound spinach (preferably Taiwanese spinach...,"[spinach, neutral oil, garlic, Salt to taste, ...",[Wash spinach clean by soaking a few times and...,"[4, Calories: 130kcal (7%), Carbohydrates: 5g ..."


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1414 entries, 0 to 1413
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Name               1414 non-null   object
 1   Ingredients_List   1414 non-null   object
 2   Ingredients_Names  1414 non-null   object
 3   Procedure          1414 non-null   object
 4   Nutrition_Facts    1414 non-null   object
dtypes: object(5)
memory usage: 55.4+ KB


In [41]:
df.sample(3)

,Name,Ingredients_List,Ingredients_Names,Procedure,Nutrition_Facts
605,multigrain-congee,"[brown rice, brown sweet rice, black rice, bla...","[brown rice, brown sweet rice, black rice, bla...",[Soak the grains in water for 2 hours. You may...,[]
1301,perfect-baked-seasoned-fries,"[1 red onion (sliced about 1/4-inch thick), ½ ...","[red onion, whole black peppercorns, bay leaf,...","[In a small pot, add all the ingredients and s...",[4]
30,eggs-soy-bean-sauce,"[1½ tablespoons soybean sauce or hoisin sauce,...","[soybean sauce or hoisin sauce, sesame oil, li...",[Prepare the sauce by combining the soybean sa...,"[3, Calories: 265kcal (13%), Carbohydrates: 6g..."


#Save Data to CSV

In [42]:
output_file = "woks_of_life_recipes.csv"

df.to_csv(
    output_file,
    index=False,
    encoding="utf-8"
)

print(f"Output file: {output_file}")

Output file: woks_of_life_recipes.csv


In [ ]:
!ls

sample_data  woks_of_life_recipes.csv
